# Exploracao de Dados em Formato Longo - Landsat 8/9

Este notebook faz uma exploracao completa do dataset `landsat89_completo.parquet` em formato longo:
- qualidade e estrutura dos dados;
- distribuicoes espectrais por banda;
- correlacoes e redundancia entre bandas;
- indices espectrais (quando as bandas de referencia existem);
- renderizacao de imagens por `image_id` + `pixel_idx`;
- formulacao de hipoteses orientadas a modelagem.


## 1) Setup


In [ ]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import pyarrow.dataset as ds
    import pyarrow.parquet as pq
except Exception:
    ds = None
    pq = None

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## 2) Configuracao do Sensor e Caminhos


In [ ]:
SENSOR_NAME = 'Landsat 8/9'
SENSOR_KEY = 'landsat89'
DEFAULT_PARQUET_FILENAME = 'landsat89_completo.parquet'

# Ajuste conforme capacidade de memoria e tamanho do dataset.
MAX_ROWS_FOR_ANALYSIS = 2_000_000
PLOT_SAMPLE_ROWS = 250_000
IMAGE_SAMPLE_COUNT = 4
CORR_THRESHOLD = 0.95
BAND_PREFIX = 'B'

SPECTRAL_HINTS = {
    "rgb": [
        "B4",
        "B3",
        "B2"
    ],
    "red": "B4",
    "green": "B3",
    "blue": "B2",
    "nir": "B5",
    "swir": "B6",
    "swir2": "B7"
}


## 3) Funcoes auxiliares


In [ ]:
def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / 'src').exists() and (cwd / 'notebooks').exists():
        return cwd
    if cwd.name == 'notebooks' and (cwd.parent / 'src').exists():
        return cwd.parent
    return cwd


def band_sort_key(name: str):
    match = re.match(r'^([A-Za-z]+)(\d+)([A-Za-z]*)$', name)
    if not match:
        return (name, 0, '')
    prefix, number, suffix = match.groups()
    return (prefix, int(number), suffix)


def infer_band_columns(columns, prefix='B'):
    pattern = re.compile(rf'^{re.escape(prefix)}\d+[A-Za-z]*$')
    bands = [c for c in columns if pattern.match(c)]
    return sorted(bands, key=band_sort_key)


def get_parquet_columns(parquet_path: Path):
    if pq is not None:
        return list(pq.ParquetFile(parquet_path).schema.names)
    # Fallback se pyarrow nao estiver instalado no ambiente.
    return list(pd.read_parquet(parquet_path).columns)


def infer_image_shape(n_pixels: int):
    side = int(np.sqrt(n_pixels))
    if side * side == n_pixels:
        return side, side
    for h in range(side, 0, -1):
        if n_pixels % h == 0:
            return h, n_pixels // h
    return 1, n_pixels


def normalize_for_display(arr, p_low=2, p_high=98):
    arr = arr.astype(np.float32, copy=False)
    finite = np.isfinite(arr)
    if not finite.any():
        return np.zeros_like(arr, dtype=np.float32)
    low, high = np.percentile(arr[finite], [p_low, p_high])
    if high <= low:
        return np.zeros_like(arr, dtype=np.float32)
    out = (arr - low) / (high - low)
    return np.clip(out, 0, 1)


def compute_index(numerator, denominator):
    with np.errstate(divide='ignore', invalid='ignore'):
        idx = numerator / denominator
    idx = idx.replace([np.inf, -np.inf], np.nan)
    return idx


def list_high_corr_pairs(corr_matrix: pd.DataFrame, threshold=0.95):
    pairs = []
    cols = list(corr_matrix.columns)
    for i, col_i in enumerate(cols):
        for j in range(i + 1, len(cols)):
            col_j = cols[j]
            value = corr_matrix.loc[col_i, col_j]
            if np.isfinite(value) and abs(value) >= threshold:
                pairs.append((col_i, col_j, value))
    if not pairs:
        return pd.DataFrame(columns=['band_a', 'band_b', 'corr'])
    out = pd.DataFrame(pairs, columns=['band_a', 'band_b', 'corr'])
    return out.sort_values('corr', key=np.abs, ascending=False).reset_index(drop=True)


def load_image_rows(parquet_path: Path, image_id_value, columns):
    if 'image_id' not in columns:
        raise ValueError('A lista de colunas precisa conter image_id para filtrar a imagem.')

    if ds is not None:
        dataset = ds.dataset(parquet_path, format='parquet')
        filter_expr = ds.field('image_id') == image_id_value
        table = dataset.to_table(columns=columns, filter=filter_expr)
        return table.to_pandas()

    full_df = pd.read_parquet(parquet_path, columns=columns)
    return full_df[full_df['image_id'] == image_id_value].copy()


def build_image_cube(image_df: pd.DataFrame, band_cols, pixel_col='pixel_idx'):
    if image_df.empty:
        raise ValueError('A imagem selecionada nao possui pixels no DataFrame.')

    values = image_df[band_cols].to_numpy(dtype=np.float32, copy=False)

    if pixel_col in image_df.columns and image_df[pixel_col].notna().all():
        idx = image_df[pixel_col].to_numpy(dtype=np.int64, copy=False)
        total_positions = int(idx.max()) + 1
        h, w = infer_image_shape(total_positions)
        flat = np.full((h * w, len(band_cols)), np.nan, dtype=np.float32)
        valid = (idx >= 0) & (idx < (h * w))
        flat[idx[valid]] = values[valid]
    else:
        total_positions = len(image_df)
        h, w = infer_image_shape(total_positions)
        if h * w != total_positions:
            raise ValueError('Nao foi possivel inferir shape valido para a imagem.')
        flat = values

    cube = flat.reshape(h, w, len(band_cols))
    return cube, (h, w)


## 4) Carregamento e visao geral


In [ ]:
PROJECT_ROOT = resolve_project_root()
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'eda_long_format'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = PROJECT_ROOT / 'data' / 'parquet' / DEFAULT_PARQUET_FILENAME
print(f'Projeto: {PROJECT_ROOT}')
print(f'Sensor: {SENSOR_NAME}')
print(f'Arquivo Parquet padrao: {DATA_PATH}')

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Arquivo nao encontrado em {DATA_PATH}. '        'Ajuste DATA_PATH para o local correto do parquet.'
    )

all_columns = get_parquet_columns(DATA_PATH)
band_cols = infer_band_columns(all_columns, prefix=BAND_PREFIX)
meta_cols = [c for c in ['image_id', 'pixel_idx'] if c in all_columns]

if not band_cols:
    raise ValueError('Nenhuma coluna de banda encontrada. Verifique o prefixo das bandas.')

print(f'Total de colunas: {len(all_columns)}')
print(f'Total de bandas detectadas: {len(band_cols)}')
print(f'Colunas de metadados detectadas: {meta_cols}')

selected_columns = meta_cols + band_cols
df_full = pd.read_parquet(DATA_PATH, columns=selected_columns)

if MAX_ROWS_FOR_ANALYSIS and len(df_full) > MAX_ROWS_FOR_ANALYSIS:
    df_analysis = df_full.sample(MAX_ROWS_FOR_ANALYSIS, random_state=RANDOM_SEED).copy()
    print(f'Usando amostra para analise: {len(df_analysis):,} linhas de {len(df_full):,}.')
else:
    df_analysis = df_full.copy()

if PLOT_SAMPLE_ROWS and len(df_analysis) > PLOT_SAMPLE_ROWS:
    df_plot = df_analysis.sample(PLOT_SAMPLE_ROWS, random_state=RANDOM_SEED).copy()
else:
    df_plot = df_analysis.copy()

memory_mb = df_analysis.memory_usage(deep=True).sum() / (1024 ** 2)
print(f'df_full shape: {df_full.shape}')
print(f'df_analysis shape: {df_analysis.shape}')
print(f'Uso de memoria (df_analysis): {memory_mb:.2f} MB')

display(df_analysis.head())


## 5) Qualidade de dados e estatisticas descritivas


In [ ]:
overview = {
    'sensor': SENSOR_NAME,
    'n_rows_total': int(len(df_full)),
    'n_rows_analysis': int(len(df_analysis)),
    'n_columns': int(df_analysis.shape[1]),
    'n_bands': int(len(band_cols)),
    'n_images': int(df_full['image_id'].nunique()) if 'image_id' in df_full.columns else None,
}

missing_pct = df_analysis[band_cols].isna().mean().mul(100).rename('missing_pct')
zero_pct = (df_analysis[band_cols] == 0).mean().mul(100).rename('zero_pct')

band_desc = df_analysis[band_cols].describe(percentiles=[0.01, 0.1, 0.5, 0.9, 0.99]).T
band_desc.index.name = 'band'
band_summary = band_desc.reset_index()
band_summary = band_summary.rename(columns={
    'mean': 'mean',
    'std': 'std',
    'min': 'min',
    '1%': 'p01',
    '10%': 'p10',
    '50%': 'p50',
    '90%': 'p90',
    '99%': 'p99',
    'max': 'max',
})
band_summary['missing_pct'] = band_summary['band'].map(missing_pct)
band_summary['zero_pct'] = band_summary['band'].map(zero_pct)

print('Resumo geral')
print(pd.Series(overview))
display(band_summary.head(10))

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
missing_pct.sort_values(ascending=False).head(20).plot(kind='bar', ax=axes[0], title='Top 20 bandas com maior % de NaN')
zero_pct.sort_values(ascending=False).head(20).plot(kind='bar', ax=axes[1], title='Top 20 bandas com maior % de zeros')
axes[0].set_ylabel('%')
axes[1].set_ylabel('%')
plt.tight_layout()
plt.show()

if 'image_id' in df_full.columns:
    image_counts = df_full.groupby('image_id').size().rename('n_pixels').sort_values(ascending=False)
    display(image_counts.describe().to_frame('n_pixels'))

    plt.figure(figsize=(8, 4))
    image_counts.sample(min(500, len(image_counts)), random_state=RANDOM_SEED).hist(bins=40)
    plt.title('Distribuicao de pixels por imagem (amostra)')
    plt.xlabel('n_pixels')
    plt.ylabel('frequencia')
    plt.show()
else:
    image_counts = pd.Series(dtype='int64', name='n_pixels')


## 6) Distribuicao das bandas


In [ ]:
plot_band_list = band_cols[:min(12, len(band_cols))]

fig, axes = plt.subplots(len(plot_band_list), 1, figsize=(10, 2.5 * len(plot_band_list)))
if len(plot_band_list) == 1:
    axes = [axes]

for ax, band in zip(axes, plot_band_list):
    sns.histplot(df_plot[band].dropna(), bins=60, ax=ax, kde=False)
    ax.set_title(f'{SENSOR_NAME} | Histograma {band}')

plt.tight_layout()
plt.show()

box_df = df_plot[plot_band_list].melt(var_name='band', value_name='reflectance')
plt.figure(figsize=(14, 5))
sns.boxplot(data=box_df, x='band', y='reflectance', showfliers=False)
plt.title(f'{SENSOR_NAME} | Boxplot (amostra)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 7) Variabilidade por imagem


In [ ]:
if 'image_id' in df_analysis.columns:
    image_band_means = df_analysis.groupby('image_id')[band_cols].mean()
    image_band_stds = df_analysis.groupby('image_id')[band_cols].std()

    print('Shape das medias por imagem:', image_band_means.shape)
    display(image_band_means.head())

    heatmap_data = image_band_means.sample(min(20, len(image_band_means)), random_state=RANDOM_SEED)
    plt.figure(figsize=(12, max(4, 0.3 * len(heatmap_data))))
    sns.heatmap(heatmap_data, cmap='viridis')
    plt.title(f'{SENSOR_NAME} | Heatmap medias de banda por imagem (amostra)')
    plt.xlabel('Bandas')
    plt.ylabel('image_id')
    plt.tight_layout()
    plt.show()
else:
    image_band_means = pd.DataFrame()
    image_band_stds = pd.DataFrame()
    print('Coluna image_id nao disponivel; pulando analise por imagem.')


## 8) Correlacao entre bandas


In [ ]:
corr_sample = df_plot[band_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_sample, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title(f'{SENSOR_NAME} | Correlacao entre bandas (amostra)')
plt.tight_layout()
plt.show()

high_corr_pairs = list_high_corr_pairs(corr_sample, threshold=CORR_THRESHOLD)
print(f'Pares com |corr| >= {CORR_THRESHOLD}: {len(high_corr_pairs)}')
display(high_corr_pairs.head(20))


## 9) Indices espectrais


In [ ]:
hints = SPECTRAL_HINTS
index_df = pd.DataFrame(index=df_plot.index)

if hints.get('nir') in band_cols and hints.get('red') in band_cols:
    index_df['NDVI'] = compute_index(
        df_plot[hints['nir']] - df_plot[hints['red']],
        df_plot[hints['nir']] + df_plot[hints['red']]
    )

if hints.get('green') in band_cols and hints.get('nir') in band_cols:
    index_df['NDWI'] = compute_index(
        df_plot[hints['green']] - df_plot[hints['nir']],
        df_plot[hints['green']] + df_plot[hints['nir']]
    )

swir_ref = hints.get('swir2') if hints.get('swir2') in band_cols else hints.get('swir')
if hints.get('nir') in band_cols and swir_ref in band_cols:
    index_df['NBR'] = compute_index(
        df_plot[hints['nir']] - df_plot[swir_ref],
        df_plot[hints['nir']] + df_plot[swir_ref]
    )

if hints.get('nir') in band_cols and hints.get('red') in band_cols and hints.get('blue') in band_cols:
    evi_num = 2.5 * (df_plot[hints['nir']] - df_plot[hints['red']])
    evi_den = df_plot[hints['nir']] + 6 * df_plot[hints['red']] - 7.5 * df_plot[hints['blue']] + 1
    index_df['EVI'] = compute_index(evi_num, evi_den)

if index_df.shape[1] == 0:
    print('Nenhum indice calculado automaticamente (bandas de referencia indisponiveis).')
    index_summary = pd.DataFrame(columns=['index_name', 'mean', 'std', 'min', 'max'])
else:
    display(index_df.head())

    fig, axes = plt.subplots(1, index_df.shape[1], figsize=(5 * index_df.shape[1], 4))
    if index_df.shape[1] == 1:
        axes = [axes]

    for ax, idx_col in zip(axes, index_df.columns):
        sns.histplot(index_df[idx_col].dropna(), bins=60, ax=ax, kde=True)
        ax.set_title(f'{SENSOR_NAME} | {idx_col}')

    plt.tight_layout()
    plt.show()

    idx_desc = index_df.describe(percentiles=[0.01, 0.5, 0.99]).T.reset_index()
    index_summary = idx_desc.rename(columns={'index': 'index_name'})
    display(index_summary)


## 10) Renderizacao de imagens


In [ ]:
if 'image_id' not in df_full.columns:
    print('Sem image_id; renderizacao de imagens nao disponivel.')
else:
    available_ids = df_full['image_id'].dropna().unique()
    n_select = min(IMAGE_SAMPLE_COUNT, len(available_ids))
    selected_ids = pd.Series(available_ids).sample(n_select, random_state=RANDOM_SEED).tolist()

    print(f'Renderizando {len(selected_ids)} imagens...')
    render_columns = ['image_id'] + (["pixel_idx"] if 'pixel_idx' in df_full.columns else []) + band_cols

    rgb_tuple = hints.get('rgb', ())
    rgb_valid = all(b in band_cols for b in rgb_tuple)

    for image_id_value in selected_ids:
        image_rows = load_image_rows(DATA_PATH, image_id_value, render_columns)
        cube, (h, w) = build_image_cube(
            image_rows,
            band_cols=band_cols,
            pixel_col='pixel_idx' if 'pixel_idx' in image_rows.columns else None,
        )

        print(f'image_id={image_id_value} | shape={h}x{w} | pixels={len(image_rows):,}')

        bands_to_show = band_cols[:min(6, len(band_cols))]
        fig, axes = plt.subplots(1, len(bands_to_show), figsize=(4 * len(bands_to_show), 4))
        if len(bands_to_show) == 1:
            axes = [axes]

        for ax, band_name in zip(axes, bands_to_show):
            band_idx = band_cols.index(band_name)
            band_img = normalize_for_display(cube[:, :, band_idx])
            ax.imshow(band_img, cmap='gray')
            ax.set_title(band_name)
            ax.axis('off')

        plt.suptitle(f'{SENSOR_NAME} | Bandas individuais | image_id={image_id_value}')
        plt.tight_layout()
        plt.show()

        if rgb_valid:
            r = normalize_for_display(cube[:, :, band_cols.index(rgb_tuple[0])])
            g = normalize_for_display(cube[:, :, band_cols.index(rgb_tuple[1])])
            b = normalize_for_display(cube[:, :, band_cols.index(rgb_tuple[2])])
            rgb = np.dstack([r, g, b])

            plt.figure(figsize=(6, 6))
            plt.imshow(rgb)
            plt.title(f'{SENSOR_NAME} | RGB {rgb_tuple} | image_id={image_id_value}')
            plt.axis('off')
            plt.show()


## 11) Hipoteses e exportacao de artefatos


In [ ]:
hypotheses = []

if not band_summary.empty:
    top_std = band_summary.sort_values('std', ascending=False).head(3)['band'].tolist()
    hypotheses.append(
        'Bandas com maior variabilidade (' + ', '.join(top_std) + ') podem carregar maior poder discriminativo para classificacao.'
    )

if not high_corr_pairs.empty:
    sample_pairs = high_corr_pairs.head(3).apply(lambda r: f"{r['band_a']}~{r['band_b']} (r={r['corr']:.2f})", axis=1).tolist()
    hypotheses.append(
        'Existe redundancia espectral relevante em ' + '; '.join(sample_pairs) + '. Vale testar PCA/selecionador de features.'
    )

if not image_band_means.empty:
    per_image_dispersion = image_band_means.std().mean()
    if np.isfinite(per_image_dispersion) and per_image_dispersion > 0:
        hypotheses.append(
            'Ha variacao entre imagens na media espectral. Hipotese: padroes temporais/atmosfericos influenciam os sinais e exigem normalizacao por cena.'
        )

if 'NDVI' in index_df.columns:
    ndvi_mean = index_df['NDVI'].mean(skipna=True)
    ndvi_std = index_df['NDVI'].std(skipna=True)
    hypotheses.append(
        f'NDVI medio={ndvi_mean:.3f} (std={ndvi_std:.3f}). Testar mascaramento de vegetacao para reduzir ruido no alvo mineral.'
    )

if not hypotheses:
    hypotheses.append('Gerar hipoteses apos validar mapeamento de bandas de referencia para este sensor.')

print('Hipoteses iniciais')
for i, h in enumerate(hypotheses, start=1):
    print(f'{i}. {h}')

band_summary_export = band_summary.copy()
band_summary_export['sensor_key'] = SENSOR_KEY
index_summary_export = index_summary.copy()
if not index_summary_export.empty:
    index_summary_export['sensor_key'] = SENSOR_KEY

overview_path = OUTPUT_DIR / f'{SENSOR_KEY}_overview.json'
band_summary_path = OUTPUT_DIR / f'{SENSOR_KEY}_band_summary.csv'
index_summary_path = OUTPUT_DIR / f'{SENSOR_KEY}_index_summary.csv'
hypotheses_path = OUTPUT_DIR / f'{SENSOR_KEY}_hypotheses.txt'
image_counts_path = OUTPUT_DIR / f'{SENSOR_KEY}_image_counts.csv'

with open(overview_path, 'w', encoding='utf-8') as f:
    json.dump(overview, f, indent=2, ensure_ascii=False)

band_summary_export.to_csv(band_summary_path, index=False)

if not index_summary_export.empty:
    index_summary_export.to_csv(index_summary_path, index=False)
elif index_summary_path.exists():
    index_summary_path.unlink()

hypotheses_path.write_text('\n'.join(hypotheses), encoding='utf-8')

if isinstance(image_counts, pd.Series) and len(image_counts) > 0:
    image_counts.reset_index().rename(columns={'index': 'image_id'}).to_csv(image_counts_path, index=False)

print('\nArtefatos salvos em:')
print('-', overview_path)
print('-', band_summary_path)
print('-', index_summary_path, '(se houver indices)')
print('-', hypotheses_path)
if image_counts_path.exists():
    print('-', image_counts_path)


## Hipoteses para validar nas proximas etapas
- Quais bandas aumentam separacao entre classes positiva/negativa apos normalizacao?
- O comportamento dos indices (NDVI/NDWI/NBR/EVI) muda por `image_id` ou por grupo temporal?
- Existe um subconjunto pequeno de bandas com desempenho comparavel ao conjunto completo?
- A reconstrução espacial das imagens indica artefatos (faixas, no-data, saturacao) que precisam ser filtrados?
